# Correlation Power Analysis (Brier et al. 2004) - Evolution Plot

In [1]:
%load_ext autoreload
%autoreload 2

import os
import random

import lascar
import numpy as np
import plotly.graph_objects as pgo
from cwtoolbox import CaptureDevice

In [2]:
class EvolutionOutputMethod(lascar.ScoreProgressionOutputMethod):
    def _finalize(self):
        scores = np.array(next(iter(self.scores.values()))).T
        fig = pgo.Figure()
        for guess, score in zip(self.engines[0]._guess_range, scores):
            fig.add_trace(pgo.Scatter(x=self.steps, y=score, name=str(guess)))
        fig.show()

In [3]:
capture_device = CaptureDevice.create("CWLITEXMEGA")
capture_device.compile(file=os.path.abspath("../lecture_3/sbox_lookup.c"))
capture_device.flash()
data = capture_device.capture(
    number_of_traces=1000,
    input=lambda _: random.randbytes(16)
)

c:\work\securecoding_ws2526\.venv\Lib\site-packages\chipwhisperer\capture\trace\TraceWhisperer.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources # type: ignore


XMEGA Programming flash...
XMEGA Reading flash...
Verified flash OK, 2553 bytes


100%|██████████| 1000/1000 [00:18<00:00, 53.37it/s]


In [4]:
def selection_function(value, guess):
    return lascar.hamming(lascar.tools.aes.sbox[value["input"][2] ^ guess])


trace = lascar.TraceBatchContainer(data["trace"], data)
engine = lascar.CpaEngine(
    name="cpa",
    selection_function=selection_function,
    guess_range=range(256),
)

session = lascar.Session(
    trace,
    engine=engine,
    output_method=EvolutionOutputMethod(engine),
    output_steps=range(0, len(data), 5),
    progressbar=False,
)
session.run(batch_size="auto")

2026-01-09 10:00:56,359 - lascar.session - INFO - Session Session: 1000 traces, 3 engines, batch_size=1233256, leakage_shape=(1376,)
INFO:lascar.session:Session Session: 1000 traces, 3 engines, batch_size=1233256, leakage_shape=(1376,)
c:\work\securecoding_ws2526\.venv\Lib\site-packages\lascar\engine\cpa_engine.py:89: RuntimeWarning: invalid value encountered in divide
  return np.nan_to_num(numerator / denominator)


In [5]:
from cwtoolbox.analyze import cpa

In [6]:
def selection_function(value, guess):
    return lascar.hamming(lascar.tools.aes.sbox[value["input"][0] ^ guess])

cpas = cpa(trace, selection_functions={"": selection_function}, guess_range=range(256))

fig = pgo.Figure()
for guess, score in cpas[0][1]:
    fig.add_trace(pgo.Scatter(y=np.abs(score), name=str(guess)))
fig.show()